# Baseline de riesgo crediticio sin variables posteriores a la decisión

Este notebook implementa las secciones 9–11 de `proposal.md`. La predicción se sitúa después de recibir la solicitud y consultar el historial crediticio, pero antes de la aprobación, la tasa y el desembolso. Usa solo préstamos históricamente aprobados cuyo estado final es `Fully Paid` o `Charged Off`; las métricas no representan automáticamente a las solicitudes rechazadas ni a los préstamos todavía `Current`.

Datos: [`accepted_2007_to_2018Q4.csv.gz`](../data/Readme.md). Las nueve variables son una selección provisional para un baseline numérico, no una selección óptima demostrada.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score,
    precision_recall_curve, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
FEATURES = (
    'annual_inc', 'dti', 'open_acc', 'pub_rec', 'revol_bal',
    'total_acc', 'fico_range_low', 'delinq_2yrs', 'inq_last_6mths',
)
DATA_NAME = 'accepted_2007_to_2018Q4.csv.gz'
candidate_paths = [root / 'data' / DATA_NAME for root in (Path.cwd(), Path.cwd().parent)]
data_path = next((path for path in candidate_paths if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError(f'Falta data/{DATA_NAME}. Consulta data/Readme.md.')
print(f'Archivo de entrada: {data_path.name}')

Archivo de entrada: accepted_2007_to_2018Q4.csv.gz


## Población y variables

Se leen únicamente la etiqueta y las nueve columnas necesarias. `Charged Off` es la clase positiva (1). Los faltantes se conservan hasta ajustar la imputación dentro del pipeline.

In [2]:
data = pd.read_csv(
    data_path, compression='gzip', usecols=[*FEATURES, 'loan_status'],
    dtype={'loan_status': 'category'}, low_memory=False,
)
data = data.loc[data['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
X = data.loc[:, list(FEATURES)].apply(pd.to_numeric, errors='coerce')
y = data['loan_status'].eq('Charged Off').astype('int8')

print(f'Préstamos con resultado definitivo: {len(y):,}')
print(f'Charged Off: {int(y.sum()):,} ({y.mean():.2%})')
print('Porcentaje de nulos por predictor:')
print((X.isna().mean() * 100).round(3).to_string())
del data

Préstamos con resultado definitivo: 1,345,310
Charged Off: 268,559 (19.96%)
Porcentaje de nulos por predictor:
annual_inc        0.000
dti               0.028
open_acc          0.000
pub_rec           0.000
revol_bal         0.000
total_acc         0.000
fico_range_low    0.000
delinq_2yrs       0.000
inq_last_6mths    0.000


## Entrenamiento, validación y prueba

Se reserva 20 % para prueba. El 80 % restante se divide en entrenamiento y validación (80/20), dando 64/16/20 en total. Ambas divisiones son estratificadas y usan la misma semilla. La prueba no participa en la elección de variables, transformaciones ni umbral.

In [3]:
X_desarrollo, X_prueba, y_desarrollo, y_prueba = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED,
)
X_entrenamiento, X_validacion, y_entrenamiento, y_validacion = train_test_split(
    X_desarrollo, y_desarrollo, test_size=0.20,
    stratify=y_desarrollo, random_state=SEED,
)
del X, y, X_desarrollo, y_desarrollo

for nombre, etiquetas in (
    ('Entrenamiento', y_entrenamiento),
    ('Validación', y_validacion),
    ('Prueba', y_prueba),
):
    print(f'{nombre}: {len(etiquetas):,} préstamos; Charged Off {etiquetas.mean():.2%}')

Entrenamiento: 860,998 préstamos; Charged Off 19.96%
Validación: 215,250 préstamos; Charged Off 19.96%
Prueba: 269,062 préstamos; Charged Off 19.96%


## Regresión logística

La mediana, el escalado y los coeficientes se ajustan solo con entrenamiento. No se ponderan clases en este primer baseline: el efecto de otras configuraciones se comparará después en validación.

In [4]:
baseline = Pipeline([
    ('imputacion', SimpleImputer(strategy='median')),
    ('escalado', StandardScaler()),
    ('modelo', LogisticRegression(max_iter=1000, random_state=SEED)),
])
baseline.fit(X_entrenamiento, y_entrenamiento)
prob_validacion = baseline.predict_proba(X_validacion)[:, 1]
print(f'Iteraciones del modelo: {baseline.named_steps["modelo"].n_iter_[0]}')

Iteraciones del modelo: 9


## Umbral de decisión

Como no se definió un costo de falsos positivos frente a falsos negativos, se escoge de forma provisional el umbral que maximiza F1 en validación. Es una referencia técnica, no una política para aprobar préstamos. El umbral queda fijo antes de evaluar prueba.

In [5]:
precision_curva, recall_curva, umbrales = precision_recall_curve(
    y_validacion, prob_validacion,
)
denominador = precision_curva[:-1] + recall_curva[:-1]
f1_curva = np.divide(
    2 * precision_curva[:-1] * recall_curva[:-1], denominador,
    out=np.zeros_like(denominador), where=denominador > 0,
)
indice_mejor = int(np.argmax(f1_curva))
umbral = float(umbrales[indice_mejor])
print(f'Umbral elegido en validación: {umbral:.4f}; F1: {f1_curva[indice_mejor]:.4f}')

Umbral elegido en validación: 0.1923; F1: 0.3729


## Evaluación

Aquí PR-AUC se calcula como *average precision* (AP). Se muestran métricas de validación y de prueba con el mismo umbral. Los resultados del notebook de exploración no son directamente comparables porque usan otras variables y otra partición.

In [6]:
def calcular_metricas(etiquetas, probabilidades, umbral_fijo):
    predicciones = (probabilidades >= umbral_fijo).astype('int8')
    return {
        'ROC-AUC': roc_auc_score(etiquetas, probabilidades),
        'PR-AUC (AP)': average_precision_score(etiquetas, probabilidades),
        'recall': recall_score(etiquetas, predicciones),
        'precisión': precision_score(etiquetas, predicciones, zero_division=0),
        'F1': f1_score(etiquetas, predicciones),
    }

prob_prueba = baseline.predict_proba(X_prueba)[:, 1]
resultados = pd.DataFrame(
    [calcular_metricas(y_validacion, prob_validacion, umbral),
     calcular_metricas(y_prueba, prob_prueba, umbral)],
    index=['Validación', 'Prueba'],
)
print(resultados.round(4).to_string())
tn, fp, fn, tp = confusion_matrix(
    y_prueba, (prob_prueba >= umbral).astype('int8'),
).ravel()
print(f'Prueba: TN={tn:,}, FP={fp:,}, FN={fn:,}, TP={tp:,}')

            ROC-AUC  PR-AUC (AP)  recall  precisión      F1
Validación   0.6326       0.2853  0.6838     0.2564  0.3729
Prueba       0.6325       0.2860  0.6822     0.2556  0.3719
Prueba: TN=108,637, FP=106,713, FN=17,068, TP=36,644


## Alcance de estos resultados

Esta es una evaluación aleatoria estratificada entre préstamos aprobados y resueltos, no una validación prospectiva. La comprobación temporal de la sección 10 de `proposal.md` queda pendiente porque los préstamos recientes aún no tienen suficientes resultados definitivos. Tampoco se ha demostrado que estas nueve variables sean las mejores.